# Lab type: review
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Datasets and DataLoaders
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## Part 1: Custom Dataset

In [ ]:
class TabularDataset(Dataset):
    """Dataset for tabular data loaded lazily from a pre-split array."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# Synthetic data
X_train = np.random.randn(8000, 20).astype(np.float32)
y_train = (X_train[:, 0] > 0).astype(np.float32)

X_val = np.random.randn(2000, 20).astype(np.float32)
y_val = (X_val[:, 0] > 0).astype(np.float32)

train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")
print(f"Single item shapes: X={train_dataset[0][0].shape}, y={train_dataset[0][1].shape}")

**Question 1:** This Dataset loads all data into memory in `__init__`. When is this approach preferable, and when would you want to load each sample lazily in `__getitem__` instead?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**When to load eagerly in `__init__`:** If the entire dataset fits in RAM, loading everything upfront avoids repeated I/O on each training iteration and keeps `__getitem__` fast. This is the right default for small-to-medium tabular or numerical datasets.

**When to load lazily in `__getitem__`:** When the dataset is too large for RAM (e.g. a full ImageNet-scale image folder), or when loading is expensive only for a subset of samples (remote object storage, DICOM decoding, high-resolution video). Lazy loading also allows on-the-fly transforms without materialising transformed data.

</details>

## Part 2: DataLoader Configuration

In [ ]:
# Training DataLoader: shuffle, drop_last=True (avoids batch-size-1 for BatchNorm)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,       # 0 = single-threaded (safe for notebooks)
    pin_memory=False,    # False: no CUDA available in this environment
    drop_last=True       # Drop final incomplete batch during training
)

# Validation DataLoader: no shuffle, no drop_last (evaluate all samples)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False      # Must evaluate all validation samples
)

# Inspect one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Batch shapes: X={X_batch.shape}, y={y_batch.shape}")
print(f"Train batches: {len(train_loader)} (from {len(train_dataset)} samples, drop_last=True)")
print(f"Val batches: {len(val_loader)} (from {len(val_dataset)} samples, drop_last=False)")

**Question 2:** The training DataLoader uses `drop_last=True` but the validation DataLoader uses `drop_last=False`. Why is this the correct configuration? What would go wrong if `drop_last=True` were applied to validation?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Training — `drop_last=True`:** A partial final batch has fewer samples than the rest. BatchNorm's batch-level mean/variance estimates are noisier with fewer samples, and the partial batch receives disproportionate gradient weight (it updates the model but covers fewer examples). Dropping it keeps statistics consistent across batches.

**Validation — `drop_last=False`:** Every validation sample must be evaluated to produce an unbiased metric. Dropping the last batch silently excludes samples; if those samples are non-randomly distributed (e.g. a sorted dataset), the metric is biased.

</details>

**Question 3:** `num_workers=0` is set here because we're in a notebook environment. In a production GPU training run with a fast SSD, what value of `num_workers` would you start with, and how would you tune it? What is the risk of setting it too high?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Starting value:** `num_workers = number of physical CPU cores / 2` is a common starting point (e.g. 4 on an 8-core machine). Profile GPU utilisation — if the GPU sits idle between batches, the bottleneck is data loading; increase `num_workers`. If system RAM usage climbs sharply or training slows, decrease it.

**Risk of setting too high:** Each worker is a separate process that gets a full copy of the Dataset object in memory. Beyond a point, the OS scheduler overhead and memory duplication cost exceeds the parallel I/O benefit. On some systems (macOS with the default `spawn` multiprocessing start method) very high worker counts can also cause deadlocks with certain DataLoader configurations.

</details>

## Part 3: pin_memory

In [ ]:
# Demonstrating pin_memory behaviour
# pin_memory=True is only beneficial when training on a CUDA GPU
# It pins host memory so the CUDA DMA engine can transfer data directly without staging

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
print()

if cuda_available:
    # With CUDA: pin_memory speeds up host→GPU transfers
    pinned_loader = DataLoader(train_dataset, batch_size=64, pin_memory=True)
    batch_X, _ = next(iter(pinned_loader))
    print(f"Pinned memory: {batch_X.is_pinned()}")
    print("Benefit: GPU can DMA-transfer directly, overlapping data transfer with compute")
else:
    print("pin_memory=True on CPU: wastes memory, provides no benefit")
    print("AI tools often set pin_memory=True regardless of hardware — check this.")

**Question 4:** An AI tool generated a DataLoader with `pin_memory=True` for your training script. Your production training server has 4 NVIDIA A100 GPUs. Should you keep this setting? What if you're debugging locally on a MacBook with an M2 chip?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**A100 GPUs — keep `pin_memory=True`:** CUDA GPUs use DMA (direct memory access) to copy tensors from CPU to GPU. Pinned (page-locked) memory allows the DMA engine to transfer data without CPU involvement, overlapping the transfer with computation. On a multi-GPU server this is a meaningful throughput gain.

**M2 MacBook — set `pin_memory=False`:** Apple Silicon uses a unified memory architecture where the CPU and GPU share the same physical memory pool. There is no PCIe bus to cross, so pinning provides no benefit. PyTorch's MPS backend will either ignore the flag or issue a warning; leaving it `True` is harmless but misleading — remove it to avoid confusion.

</details>

## Part 4: Custom Image Transforms

In [ ]:
# Standard torchvision transform pipeline for training images
# (ImageFolder / custom image Dataset would use this)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),                          # [0,255] uint8 → [0.0,1.0] float32
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],                 # ImageNet mean per channel
        std=[0.229, 0.224, 0.225]                   # ImageNet std per channel
    )
])

# Validation transforms: NO augmentation — only resize/crop and normalise
val_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Train transform pipeline:", train_transform)
print()
print("Val transform pipeline:", val_transform)

**Question 5:** The validation transform deliberately omits `RandomHorizontalFlip` and `RandomCrop`. Why? When would test-time augmentation (applying random transforms during inference and averaging predictions) be appropriate?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Why omit augmentation on validation:** Augmentation introduces randomness — `RandomHorizontalFlip` and `RandomCrop` produce a different transformed image on each call. Applied during validation, this means the same model produces different metric values across runs, making it impossible to reliably compare epochs or detect overfitting. Validation should measure the model's performance on a fixed, representative distribution.

**When TTA is appropriate:** Test-time augmentation (applying random transforms at inference and averaging predictions) is useful when maximising predictive accuracy on a final held-out test set matters more than reproducibility — e.g. competition submissions or ensemble inference. It is not appropriate during routine training-loop validation, where metric consistency is the priority.

</details>

## Summary

> **Final check:** Answer in one sentence each.

1. When should you use `drop_last=True`, and for which split?
2. When does `pin_memory=True` provide a benefit?
3. What is the safe starting value for `num_workers` in a notebook, and why?


<details>
<summary>🔑 Reveal summary answers</summary>

1. **`drop_last=True` for training only** — use it on the training split to prevent partial final batches from distorting BatchNorm statistics and gradient weighting; never on validation where every sample must be evaluated.

2. **`pin_memory=True` benefits CUDA GPUs** — it speeds up host-to-device transfer via DMA on machines with a discrete GPU over PCIe; it provides no benefit (and can be set `False`) on unified-memory hardware like Apple Silicon.

3. **`num_workers=0` in notebooks** — the safe default because notebook environments use `fork`-incompatible multiprocessing on some platforms; start with 0 and increase only in standalone training scripts after verifying no deadlock or memory issues.

</details>